# AutoML con PyCaret

Parte del ítem 6 del trabajo práctico de predicción de lluvia. Corre aparte del notebook principal
porque PyCaret 3.3.2, la última versión, sólo funciona con Python 3.9 a 3.11 y con scikit-learn 1.4, y
el notebook principal usa Python 3.12 y scikit-learn 1.9. Usa su propio entorno, `.venv-pycaret` (cómo
armarlo está en el README de esta carpeta).

Recibe los datos ya preprocesados por el `Pipeline` del notebook principal, ajustado sólo con
entrenamiento, y devuelve tres tablas en `resultados/`: la comparación de modelos en validación cruzada, la
probabilidad de lluvia que cada uno de los mejores le asigna a cada día de validación, y la de AdaBoost. La lectura de
los resultados está en el notebook principal.

In [1]:
# Bibliotecas y la misma semilla que el notebook principal
import numpy as np
import pandas as pd
from sklearn.metrics import precision_score, recall_score
from pycaret.classification import add_metric, compare_models, create_model, predict_model, pull, setup

SEMILLA = 42
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

In [2]:
# Entrenamiento y validación preprocesados, como los exporta el notebook principal
entrenamiento = pd.read_csv("datos/entrenamiento.csv.gz")
validacion = pd.read_csv("datos/validacion.csv.gz")
anio_entrenamiento = entrenamiento.pop("anio").to_numpy()
print(f"Entrenamiento: {entrenamiento.shape}  |  Validación: {validacion.shape}  |  Faltantes: {entrenamiento.isna().sum().sum() + validacion.isna().sum().sum()}")
print(f"Proporción de lluvia: entrenamiento {entrenamiento['RainTomorrow'].mean():.3f}, validación {validacion['RainTomorrow'].mean():.3f}")

Entrenamiento: (98004, 29)  |  Validación: (14016, 29)  |  Faltantes: 0
Proporción de lluvia: entrenamiento 0.224, validación 0.218


## Mismos folds y mismo criterio que la búsqueda de hiperparámetros

Para que los modelos de PyCaret se puedan comparar con la regresión logística, tienen que medirse igual
que en la búsqueda de hiperparámetros. Los folds son los mismos cinco años, cada uno validado con un
modelo entrenado con todos los anteriores. PyCaret acepta cualquier objeto con los métodos `split` y
`get_n_splits`, como los de scikit-learn, así que armamos uno que devuelve esos pares de índices.

La métrica es la misma: el recall de la lluvia con el umbral que la lleva más alto sin bajar la
precisión de 0,5, elegido en cada fold. PyCaret mide por defecto con el umbral 0,5 y ordena por accuracy,
que es justo lo que descartamos, así que la agregamos con `add_metric` y ordenamos por ella.

Hay una diferencia con la búsqueda de hiperparámetros, y es un costo de usar PyCaret con los datos ya
preprocesados: el imputador y el escalador se ajustaron con todo entrenamiento, también con el año que
cada fold valida. Para medir cuánto pesa, la regresión logística de PyCaret tiene que dar casi el mismo
recall por año que la búsqueda, donde el preprocesamiento sí se ajustaba en cada fold. El recall por año de la búsqueda lo exporta el notebook principal, junto con los datos.

In [3]:
class FoldsPorAnio:
    """Folds que validan cada uno un año y entrenan con todos los anteriores, con la interfaz de los de scikit-learn."""

    def __init__(self, anio, anios_validacion):
        self.anio = anio
        self.anios_validacion = anios_validacion

    def split(self, X=None, y=None, groups=None):
        for anio in self.anios_validacion:
            yield np.flatnonzero(self.anio < anio), np.flatnonzero(self.anio == anio)

    def get_n_splits(self, X=None, y=None, groups=None):
        return len(self.anios_validacion)

In [4]:
def recall_con_piso(y, probabilidad, **kwargs):
    """Recall de la lluvia con el umbral entre 0,05 y 0,95 que lo hace máximo con precisión de al menos 0,5; 0 si ninguno llega al piso."""
    umbrales = np.round(np.arange(0.05, 0.96, 0.01), 2)
    recalls = [recall_score(y, probabilidad >= u) for u in umbrales if precision_score(y, probabilidad >= u, zero_division=0) >= 0.5]
    return max(recalls, default=0.0)

In [5]:
# Experimento de PyCaret sin su preprocesamiento, con validación como conjunto de prueba y los folds por año
folds = FoldsPorAnio(anio_entrenamiento, range(2010, 2015))
experimento = setup(data=entrenamiento, target="RainTomorrow", test_data=validacion, preprocess=False, index=False,
                    fold_strategy=folds, data_split_shuffle=False, fold_shuffle=False,
                    session_id=SEMILLA, n_jobs=-1, html=False, verbose=False)
add_metric("recall_piso", "Recall con precisión ≥ 0,5", recall_con_piso, target="pred_proba")
pd.DataFrame({anio: {"filas entrenamiento": len(entrena), "filas validación": len(valida)}
              for anio, (entrena, valida) in zip(folds.anios_validacion, folds.split())}).T

,filas entrenamiento,filas validación
2010,18507,16124
2011,34631,15032
2012,49663,14914
2013,64577,16035
2014,80612,17112


In [6]:
# Si los folds y la métrica son los mismos, la regresión logística da casi el recall por año de la búsqueda de hiperparámetros
create_model("lr", round=6, verbose=False)
recall_lr = pd.DataFrame({"PyCaret": pull()["Recall con precisión ≥ 0,5"].iloc[:5].to_numpy(),
                          "búsqueda de hiperparámetros": pd.read_csv("datos/recall_busqueda.csv", index_col=0)["recall"].to_numpy()},
                         index=pd.Index(folds.anios_validacion, name="año validado"))
recall_lr.assign(diferencia=recall_lr["PyCaret"] - recall_lr["búsqueda de hiperparámetros"]).round(6)

,PyCaret,búsqueda de hiperparámetros,diferencia
año validado,,,
2010,0.845060,0.844806,0.000254
2011,0.825651,0.825922,-0.000271
2012,0.823671,0.824272,-0.000601
2013,0.792973,0.793264,-0.000291
2014,0.760596,0.761455,-0.000859


Los cinco años coinciden con la búsqueda de hiperparámetros a menos de 0,001, así que los folds son
los mismos y ajustar el preprocesamiento con todo entrenamiento casi no cambia la medición.

## Comparación de modelos

In [7]:
# Todos los modelos de clasificación que PyCaret prueba por defecto, ordenados por el recall con el piso de precisión
compare_models(sort="Recall con precisión ≥ 0,5", verbose=False)
comparacion = pull()
comparacion.to_csv("resultados/comparacion.csv")
comparacion

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,"Recall con precisión ≥ 0,5",TT (Sec)
lightgbm,Light Gradient Boosting Machine,0.8586,0.8917,0.5547,0.7543,0.6391,0.5536,0.5638,0.8505,0.640
et,Extra Trees Classifier,0.8544,0.8854,0.5055,0.7715,0.6107,0.5258,0.5436,0.8397,1.276
rf,Random Forest Classifier,0.8567,0.8841,0.5201,0.7716,0.6213,0.5369,0.5528,0.8357,2.330
gbc,Gradient Boosting Classifier,0.8542,0.8813,0.5313,0.7508,0.6221,0.5349,0.5473,0.8242,6.714
lr,Logistic Regression,0.8476,0.8723,0.5291,0.7221,0.6105,0.5185,0.5283,0.8096,0.444
lda,Linear Discriminant Analysis,0.8461,0.8702,0.5398,0.7090,0.6128,0.5188,0.5264,0.8031,0.340
qda,Quadratic Discriminant Analysis,0.8187,0.8529,0.6395,0.5913,0.6144,0.4959,0.4966,0.7757,0.402
nb,Naive Bayes,0.7801,0.8359,0.7076,0.5101,0.5928,0.4472,0.4585,0.7185,0.302
knn,K Neighbors Classifier,0.8324,0.8212,0.4928,0.6780,0.5706,0.4697,0.4790,0.6735,3.634
ada,Ada Boost Classifier,0.8457,0.8669,0.5184,0.7200,0.6027,0.5099,0.5206,0.5184,1.540


In [8]:
# Los tres mejores y la regresión logística, otra vez, para guardar su recall en cada año
modelos_validacion = {}
recall_anio = {}
for nombre in [*comparacion.index[:3].drop("lr", errors="ignore"), "lr"]:
    modelos_validacion[nombre] = create_model(nombre, verbose=False)
    recall_anio[nombre] = pull()["Recall con precisión ≥ 0,5"].iloc[:5].to_numpy()
recall_anio = pd.DataFrame(recall_anio, index=pd.Index(folds.anios_validacion, name="año validado"))
recall_anio.to_csv("resultados/recall_por_anio.csv")
recall_anio.round(3)

,lightgbm,et,rf,lr
año validado,,,,
2010,0.864,0.861,0.845,0.845
2011,0.859,0.843,0.848,0.826
2012,0.862,0.847,0.842,0.824
2013,0.840,0.845,0.839,0.793
2014,0.826,0.802,0.804,0.761


In [9]:
# Probabilidad de lluvia que cada uno de esos modelos le asigna a cada día de validación
probabilidades = pd.DataFrame({"RainTomorrow": validacion["RainTomorrow"].to_numpy()})
for nombre, modelo in modelos_validacion.items():
    prediccion = predict_model(modelo, raw_score=True, verbose=False)
    probabilidades[nombre] = prediccion["prediction_score_1"].to_numpy()
probabilidades.to_csv("resultados/probabilidades_validacion.csv.gz", index=False)
print(f"Modelos: {list(modelos_validacion)}  |  Días: {len(probabilidades)}  |  "
      f"Mismo orden que validación: {(probabilidades['RainTomorrow'].to_numpy() == prediccion['RainTomorrow'].to_numpy()).all()}")

Modelos: ['lightgbm', 'et', 'rf', 'lr']  |  Días: 14016  |  Mismo orden que validación: True


In [10]:
# Probabilidad de lluvia de AdaBoost en cada día de validación, para revisar por qué su recall con el piso es el del umbral 0,5
prediccion_ada = predict_model(create_model("ada", verbose=False), raw_score=True, verbose=False)
pd.DataFrame({"ada": prediccion_ada["prediction_score_1"].to_numpy()}).to_csv("resultados/probabilidades_adaboost.csv.gz", index=False)
print(f"Días: {len(prediccion_ada)}  |  Mismo orden que validación: {(prediccion_ada['RainTomorrow'].to_numpy() == validacion['RainTomorrow'].to_numpy()).all()}")

Días: 14016  |  Mismo orden que validación: True
